# 2. Text Preprocessing and Feature Extraction
## AG News Dataset Preprocessing

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.preprocessing import TextPreprocessor, FeatureExtractor
from src.utils import save_model

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Load Data

In [2]:
train_df = pd.read_csv('../data/processed/train_explored.csv')
test_df = pd.read_csv('../data/processed/test_explored.csv')

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 120000, Test: 7600


## Text Cleaning

In [3]:
# Initialize preprocessor
preprocessor = TextPreprocessor(remove_stopwords=True, lemmatize=True)

# Example preprocessing
sample_text = train_df['text'].iloc[0]
print("Original:")
print(sample_text)
print("\nCleaned:")
print(preprocessor.preprocess(sample_text))

Original:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Cleaned:
wall st bear claw back black reuters reuters shortsellers wall street dwindlingband ultracynics seeing green


In [4]:
# Apply to all data
print("Preprocessing training data...")
train_df = preprocessor.preprocess_dataframe(train_df)

print("Preprocessing test data...")
test_df = preprocessor.preprocess_dataframe(test_df)

print("Done!")

Preprocessing training data...
Preprocessing test data...
Done!


## Split Training Data into Train (85%) and Validation (15%) sets

In [5]:
from sklearn.model_selection import train_test_split

# Split train data into train (85%) and validation (15%)
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['class']
)

print(f"Original training size: {len(train_df)}")
print(f"New training size: {len(train_indices)} (85%)")
print(f"Validation size: {len(val_indices)} (15%)")
print(f"Test size: {len(test_df)}")

Original training size: 120000
New training size: 102000 (85%)
Validation size: 18000 (15%)
Test size: 7600


## Feature Extraction - TF-IDF

In [6]:
# Extract TF-IDF features from FULL training data first
tfidf_extractor = FeatureExtractor(method='tfidf', max_features=5000)

X_full_tfidf = tfidf_extractor.fit_transform(train_df['cleaned_text'])
X_test_tfidf = tfidf_extractor.transform(test_df['cleaned_text'])

# Now split into train and validation
X_train_tfidf = X_full_tfidf[train_indices]
X_val_tfidf = X_full_tfidf[val_indices]

print(f"TF-IDF features shape:")
print(f"  Train: {X_train_tfidf.shape}")
print(f"  Validation: {X_val_tfidf.shape}")
print(f"  Test: {X_test_tfidf.shape}")

TF-IDF features shape:
  Train: (102000, 5000)
  Validation: (18000, 5000)
  Test: (7600, 5000)


## Feature Extraction - Count Vectorizer (Bag of Words)

In [7]:
# Extract count features from FULL training data first
count_extractor = FeatureExtractor(method='count', max_features=5000)

X_full_count = count_extractor.fit_transform(train_df['cleaned_text'])
X_test_count = count_extractor.transform(test_df['cleaned_text'])

# Now split into train and validation
X_train_count = X_full_count[train_indices]
X_val_count = X_full_count[val_indices]

print(f"Count (BoW) features shape:")
print(f"  Train: {X_train_count.shape}")
print(f"  Validation: {X_val_count.shape}")
print(f"  Test: {X_test_count.shape}")

Count (BoW) features shape:
  Train: (102000, 5000)
  Validation: (18000, 5000)
  Test: (7600, 5000)


## Save Preprocessed Data and Vectorizers

In [8]:
import joblib

# Save data
train_df.to_csv('../data/processed/train_preprocessed.csv', index=False)
test_df.to_csv('../data/processed/test_preprocessed.csv', index=False)

# Save vectorizers
save_model(tfidf_extractor, '../models/tfidf_vectorizer.pkl')
save_model(count_extractor, '../models/count_vectorizer.pkl')

# Save TF-IDF features
joblib.dump(X_train_tfidf, '../data/processed/X_train_tfidf.pkl')
joblib.dump(X_val_tfidf, '../data/processed/X_val_tfidf.pkl')
joblib.dump(X_test_tfidf, '../data/processed/X_test_tfidf.pkl')

# Save Count (BoW) features
joblib.dump(X_train_count, '../data/processed/X_train_count.pkl')
joblib.dump(X_val_count, '../data/processed/X_val_count.pkl')
joblib.dump(X_test_count, '../data/processed/X_test_count.pkl')

# Save labels
y_train = train_df['class'].values[train_indices]
y_val = train_df['class'].values[val_indices]
y_test = test_df['class'].values

joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_val, '../data/processed/y_val.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

print("\nAll preprocessed data saved!")
print(f"\nSaved splits:")
print(f"  Train: {X_train_tfidf.shape[0]} samples")
print(f"  Validation: {X_val_tfidf.shape[0]} samples")
print(f"  Test: {X_test_tfidf.shape[0]} samples")

Model saved to ../models/tfidf_vectorizer.pkl
Model saved to ../models/count_vectorizer.pkl

All preprocessed data saved!

Saved splits:
  Train: 102000 samples
  Validation: 18000 samples
  Test: 7600 samples
